# Import required libraries

In [ ]:
%pip install groq fastmcp pyngrok

In [ ]:
import os
import json
from getpass import getpass

# Required library
from groq import AsyncGroq
from fastmcp import Client
from pyngrok import ngrok, conf

# Groq (Free-tier AI API)
Get your api key here: https://console.groq.com/keys

In [ ]:
groq = AsyncGroq(api_key=os.getenv("GROQ_API_KEY") or getpass("Groq API key: "))

MODEL = "openai/gpt-oss-120b"

# Ask Data with Public MCP

In [ ]:
async def ask_data_public_mcp(question: str):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a weather reporter"
                "Use the available tools when needed to answer questions"
            ),
        },
        {
            "role": "user",
            "content": question 
        },
    ]

    response = await groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[
            {
                "type": "mcp",
                "server_label": "weather_intelligence",
                "server_url": "https://api.tomorrow.io/v4/weather-intelligence/mcp",
                "require_approval": "never",
            }
        ],
    )

    # Uncomment this to see the raw output
    # print(response.model_dump_json(indent=2))

    return response.choices[0].message.content

In [ ]:
print(await(ask_data_public_mcp("How is the bangkok weather now ?")))

# Prepare system & user prompt

In [ ]:
def get_prompt(question: str):
    return [
        {
            "role": "system",
            "content": (
                "You are a data science assistant. "
                "Use the available tools when needed to answer questions "
                "about the dataset. Never invent computed results."
            ),
        },
        {
            "role": "user",
            "content": question 
        },
    ]

# Ask Data with Local MCP (import from `mcp_server.py`)

In [ ]:
# Local library
from mcp_server import mcp

In [ ]:
async def ask_data_local_import_mcp(question: str):
    messages = get_prompt(question)
    # Local MCP client: tool execution happens in this notebook.
    async with Client(mcp) as client:
        mcp_tools = await client.list_tools()

        tools = [
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or tool.name,
                    "parameters": tool.input_schema,
                },
            }
            for tool in mcp_tools
        ]

        allowed_names = {tool.name for tool in mcp_tools}

        ### 
        # Loop notes
        # 1. [sys, user ] -> ask = msg
        # 2. [sys, user, msg ] -> call tool = tool_output
        # 3. [sys, user, msg, tool_output ] -> answer !
        ###

        for _ in range(5):  # Limit tool-calling rounds
            response = await groq.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto",
            )

            msg = response.choices[0].message
            messages.append(msg)

            # The model decided it has enough information.
            if not msg.tool_calls:
                return msg.content

            for call in msg.tool_calls:
                name = call.function.name
                args = json.loads(call.function.arguments)

                # Only execute tools actually exposed by this MCP server.
                if name not in allowed_names:
                    raise ValueError(f"Unknown tool: {name}")

                print(f"Model selected: {name}({args})")

                try:
                    result = await client.call_tool(name, args)
                    output = (
                        result.structured_content
                        if result.structured_content is not None
                        else [
                            getattr(item, "text", str(item)) for item in result.content
                        ]
                    )
                    content = json.dumps(output, default=str)
                except Exception as e:
                    content = json.dumps({"error": str(e)})

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": content,
                    }
                )

    return "Stopped because the tool-call limit was reached."


In [ ]:
print(await(ask_data_local_import_mcp("Which province has the highest average rainfall?")))

# Ask Data with Local MCP (tunnel with `pyngrok` or local `ngrok` cli)

Also run `python mcp_server.py` on separate process

## Connect ngrok

In [ ]:
config = conf.get_default()

if not config.auth_token:
    ngrok.set_auth_token(getpass("ngrok Auth Token: "))

# Open an HTTP tunnel on port 8000
http_tunnel = ngrok.connect(8000)

# Get the public URL
public_url = http_tunnel.public_url
mcp_url = f"{public_url}/mcp"

print("Public URL:", public_url)
print("MCP URL:", mcp_url)

If you have problem with Windows pyngrok, connect manually via cmd `ngrok http 80`

In [ ]:
# public_url = "YOUR_NGROK_URL"
# mcp_url = f"{public_url}/mcp"

# print("Public URL:", public_url)
# print("MCP URL:", mcp_url)

In [ ]:
async def ask_data_local_server_mcp(question: str):
    messages = get_prompt(question)

    response = await groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[
            {
                "type": "mcp",
                "server_label": "data_science",
                "server_url": mcp_url,
                "require_approval": "never",
            }
        ],
    )

    return response.choices[0].message.content

In [ ]:
print(await(ask_data_local_server_mcp("Which province has the highest average rainfall?")))

## Disconnect ngrok

In [ ]:
ngrok.disconnect(http_tunnel.public_url)